In [11]:
import ollama
import json
import re
from pypdf import PdfReader

print("✅ Setup complete")

✅ Setup complete


In [12]:
reader = PdfReader(r"E:\PersonaForge-A-Multi-Agent-LLM-System-for-Intelligent-Interaction\me\Profile.pdf")

linkedin_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin_text += text

print("✅ LinkedIn data loaded")

✅ LinkedIn data loaded


In [13]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary_text = f.read()

print("✅ Summary loaded")

✅ Summary loaded


In [14]:
GENERATOR_MODEL = "llama3.2"
EVALUATOR_MODEL = "phi3"

NAME = "Nikith"  

SYSTEM_PROMPT = f"""
You are acting as {NAME}, an AI/Cloud Engineer.

You must answer like {NAME} based on real experience.

RULES:
- Use first person ("I", "my experience")
- Be professional and natural
- Keep answers concise
- If you don’t know, say so

--- SUMMARY ---
{summary_text}

--- LINKEDIN ---
{linkedin_text}
"""

In [15]:
EVALUATOR_PROMPT = """
You are a STRICT AI evaluator.

Evaluate the response based on:
- clarity
- correctness
- completeness
- professionalism
- instruction following
- human-like tone

IMPORTANT:
- If response is too long → simplify
- If not beginner-friendly → rewrite
- If robotic → make natural
- ALWAYS provide improved_response if possible

Return ONLY valid JSON.

Format:
{
  "score": 1-10,
  "verdict": "approve" or "improve",
  "feedback": "short feedback",
  "improved_response": "better response"
}
"""

In [16]:
def clean_response(text):
    parts = text.strip().split("\n\n")
    seen = []
    for p in parts:
        if p not in seen:
            seen.append(p)
    return "\n\n".join(seen)

In [17]:
def generator_agent(user_input, chat_history):
    user_input = user_input + "\n\nFollow the instruction strictly. Keep it short."

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + chat_history + [
        {"role": "user", "content": user_input}
    ]

    response = ollama.chat(
        model=GENERATOR_MODEL,
        messages=messages
    )

    return response["message"]["content"]

In [18]:
def evaluator_agent(user_input, response):
    short_response = response[:300]

    messages = [
        {"role": "system", "content": EVALUATOR_PROMPT},
        {"role": "user", "content": f"""
Question: {user_input}

Response: {short_response}
"""}
    ]

    eval_response = ollama.chat(
        model=EVALUATOR_MODEL,
        messages=messages
    )

    content = eval_response["message"]["content"]

    try:
        json_str = re.search(r"\{.*\}", content, re.DOTALL).group()
        return json_str
    except:
        return '{"score": 5, "verdict": "approve", "feedback": "fallback", "improved_response": ""}'

In [19]:
def multi_agent_chat(user_input, chat_history=[]):
    # Step 1: Generate
    raw_response = generator_agent(user_input, chat_history)

    # Step 2: Evaluate
    evaluation = evaluator_agent(user_input, raw_response)

    try:
        eval_json = json.loads(evaluation)
    except:
        eval_json = {
            "score": "N/A",
            "verdict": "approve",
            "feedback": "JSON parsing failed",
            "improved_response": raw_response
        }

    # Step 3: Decision Logic (SMART)
    improved = eval_json.get("improved_response", "").strip()

    if len(raw_response.split()) > 80:
        final_response = improved if improved and improved != raw_response else raw_response

    elif improved:
        final_response = improved

    else:
        final_response = raw_response

    # Step 4: Clean duplicates
    final_response = clean_response(final_response)

    # Step 5: Update history
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": final_response})

    # Step 6: Debug output
    print("\n🧠 Generator Response:\n", raw_response)
    print("\n🔍 Evaluator Output:\n", eval_json)
    print("\n✅ Final Response:\n", final_response)

    return final_response, chat_history

In [20]:
chat_history = []

reply, chat_history = multi_agent_chat(
    "can you discuss your role as a president of tht club that you been part of in college? ",
    chat_history
)

print("\n🎯 OUTPUT:\n", reply)


🧠 Generator Response:
 As the President of the Perspective Arts Club at GITAM University, I led and managed a diverse team of talented artists, fostering a collaborative and inclusive environment to nurture their growth.

My key responsibilities included planning and executing highly successful events, such as art exhibitions, workshops, and panel discussions, attracting participants from within and outside the university community. I also implemented innovative strategies to enhance the visibility and impact of the club, resulting in increased membership and participation.

Some notable achievements include:

* Organizing fundraising initiatives, securing sponsorships for financial sustainability
* Launching a mentorship program, connecting experienced artists with emerging talents
* Collaborating with student organizations and academic departments, fostering interdisciplinary projects
* Recognizing and celebrating member achievements through awards

This experience shaped my commitm